In [8]:
import time
import os
import pdb

# Función para crear automáticamente los archivos .txt usando codificación limpia
def inicializar_archivos():
    # 1. Archivo del menú de precios
    if not os.path.exists("menu_precios.txt"):
        contenido_menu = (
            "1,Famous Star,110.00\n"
            "2,Western Bacon,135.00\n"
            "3,Guacamole Burger,150.00\n"
            "4,Combo Super Star,180.00"
        )
        with open("menu_precios.txt", "w", encoding="utf-8") as f:
            f.write(contenido_menu)
            
    # 2. Archivo de reporte de ventas
    if not os.path.exists("reporte_ventas.txt"):
        with open("reporte_ventas.txt", "w", encoding="utf-8") as f:
            f.write("--- INICIO DEL REPORTE DE VENTAS DEL DIA ---\n")
            
    # 3. Archivo de historial de clientes (Inicia vacío con su encabezado para llenarse solo)
    if not os.path.exists("historial_clientes.txt"):
        with open("historial_clientes.txt", "w", encoding="utf-8") as f:
            f.write("--- REGISTRO DE CLIENTES FRECUENTES ---\n")
            
    # 4. Archivo de registro de errores (log)
    if not os.path.exists("log_errores.txt"):
        with open("log_errores.txt", "w", encoding="utf-8") as f:
            f.write("--- LOG DE ERRORES INICIALIZADO ---\n")

# Esta función limpia la pantalla negra para que el texto no se vea amontonado
def limpiar_pantalla():
    os.system('cls' if os.name == 'nt' else 'clear')

# Hacemos una pausa de unos segundos para simular que el sistema está cargando
def pantalla_carga():
    limpiar_pantalla()
    print("\nIniciando conexion con el servidor de Carl's Jr...")
    time.sleep(2)
    print("Cargando bases de datos de hamburguesas...")
    time.sleep(1.5)
    print("¡Sistema listo y cargado!\n")
    time.sleep(1)

# Guardamos los errores en el archivo de texto por si el programa falla
def registrar_error(mensaje):
    try:
        with open("log_errores.txt", "a", encoding="utf-8") as archivo:
            archivo.write("ERROR DETECTADO: " + mensaje + "\n")
    except Exception:
        pass 

# Función principal para registrar los pedidos y cobrar
def realizar_cobro(nickname, fecha_str):
    limpiar_pantalla()
    try:
        # Abrimos el archivo de texto externo para leer los precios
        with open("menu_precios.txt", "r", encoding="utf-8") as archivo_menu:
            lineas_menu = archivo_menu.readlines()
        
        # Pedimos el nombre del cliente para llamarlo al entregar su pedido
        print("--- NUEVA ORDEN ---")
        nombre_cliente = input("Nombre del cliente: ")
        
        subtotal_orden = 0.0
        detalle_ticket = ""
        comprando = True
        
        print("\n--- MENU DE HAMBURGUESAS ---")
        for linea in lineas_menu:
            datos = linea.strip().split(",")
            print(datos[0] + ". " + datos[1] + " - $" + datos[2])
        
        # Un ciclo por si el cliente quiere pedir varias cosas diferentes
        while comprando:
            op_compra = input("\nNumero de hamburguesa: ")
            
            # Ciclo para atrapar al usuario si pone letras en vez de números
            while True:
                try:
                    cantidad = int(input("Cantidad de piezas: "))
                    if cantidad > 0:
                        break 
                    else:
                        print("Por favor ingresa una cantidad mayor a 0.")
                except ValueError:
                    print("Error: Ingresaste letras. Por favor teclea un numero.")
                    registrar_error("El cajero puso letras en la cantidad.")
            
            encontrado = False
            for linea in lineas_menu:
                datos = linea.strip().split(",")
                if datos[0] == op_compra:
                    precio = float(datos[2])
                    nombre_ham = datos[1]
                    
                    subtotal_item = precio * cantidad
                    subtotal_orden = subtotal_orden + subtotal_item
                    
                    detalle_ticket += "- " + nombre_ham + " x" + str(cantidad) + " ($" + str(subtotal_item) + ")\n"
                    encontrado = True
                    print("-> ¡Agregado al pedido!")
                    break
            
            if not encontrado:
                print("Esa opcion no existe en el menu.")
            
            otra = input("\n¿Desea agregar otra hamburguesa diferente? (si/no): ")
            if otra.lower() != "si":
                comprando = False
        
        # Si sí compraron algo, pasamos a cobrar
        if subtotal_orden > 0:
            iva = subtotal_orden * 0.16
            total = subtotal_orden + iva
            
            print("\n--- PROCESO DE PAGO ---")
            print("El total a cobrar es de: $" + str(round(total, 2)))
            
            metodo = input("¿Metodo de pago? (1 para Efectivo / 2 para Tarjeta): ")
            
            if metodo == "2":
                tipo_pago = "Tarjeta"
                print("Por favor, inserte o deslice la tarjeta en la terminal...")
                time.sleep(2)
                print("Procesando...")
                time.sleep(1)
                print("¡Pago aprobado! Retirando tarjeta...")
                pago_cliente = total
                cambio = 0.0
            else:
                tipo_pago = "Efectivo"
                # Verificamos que el dinero alcance
                while True:
                    try:
                        pago_cliente = float(input("Ingrese el efectivo entregado por el cliente: $"))
                        if pago_cliente >= total:
                            break
                        else:
                            print("Efectivo insuficiente. Faltan $" + str(round(total - pago_cliente, 2)))
                    except ValueError:
                        print("Error: Ingresa un monto valido.")
                        registrar_error("El cajero puso letras al cobrar.")
                    
                cambio = pago_cliente - total
                print("\n¡Cobro exitoso! El cambio a devolver es: $" + str(round(cambio, 2)))
            
            # Armamos el diseño del ticket
            ticket_final = "\n========================================\n"
            ticket_final += "Fecha: " + fecha_str + " | Cajero: " + nickname + "\n"
            ticket_final += "Cliente a llamar: " + nombre_cliente.upper() + "\n"
            ticket_final += "----------------------------------------\n"
            ticket_final += detalle_ticket
            ticket_final += "----------------------------------------\n"
            ticket_final += "Subtotal: $" + str(round(subtotal_orden, 2)) + "\n"
            ticket_final += "IVA (16%): $" + str(round(iva, 2)) + "\n"
            ticket_final += "Total Final: $" + str(round(total, 2)) + "\n"
            ticket_final += "----------------------------------------\n"
            ticket_final += "Metodo de Pago: " + tipo_pago + "\n"
            
            if tipo_pago == "Efectivo":
                ticket_final += "Efectivo Recibido: $" + str(round(pago_cliente, 2)) + "\n"
                ticket_final += "Cambio: $" + str(round(cambio, 2)) + "\n"
            else:
                ticket_final += "Estado: APROBADO (Sin cambio)\n"
                
            ticket_final += "========================================\n"
            
            print(ticket_final)
            
            # 1. Guardamos el ticket en el archivo de ventas
            with open("reporte_ventas.txt", "a", encoding="utf-8") as archivo_ventas:
                archivo_ventas.write(ticket_final)
            
            # 2. Guardamos automáticamente al cliente ingresado en el historial de forma dinámica
            with open("historial_clientes.txt", "a", encoding="utf-8") as archivo_clientes:
                archivo_clientes.write("Cliente: " + nombre_cliente.title() + " | Estado: Frecuente / Activo\n")
                
    except FileNotFoundError:
        print("\nError: No se encontro el archivo 'menu_precios.txt' en la carpeta.")
        registrar_error("FileNotFoundError: Falta el archivo menu_precios.txt")
    except Exception as e:
        print("\nUps, ocurrio un error inesperado.")
        registrar_error(str(e))
        
    input("\nPresiona ENTER para regresar al menu principal...")

# Función para ver los tickets leídos desde el archivo externo
def imprimir_tickets():
    limpiar_pantalla()
    try:
        with open("reporte_ventas.txt", "r", encoding="utf-8") as arch_ventas:
            print("--- COPIA DE TICKETS DEL DIA ---")
            contenido = arch_ventas.read()
            if contenido == "":
                print("(No se han registrado ventas hoy)")
            else:
                print(contenido)
    except FileNotFoundError:
        print("Aviso: El archivo 'reporte_ventas.txt' no existe todavia en la carpeta.")
        
    input("\nPresiona ENTER para regresar al menu principal...")

# Función para abrir y leer cualquiera de los 4 archivos externos usando un diccionario
def area_usuario():
    limpiar_pantalla()
    print("--- AREA DE USUARIO: ARCHIVOS DEL SISTEMA ---")
    archivos_disponibles = {
        "1": "menu_precios.txt",
        "2": "reporte_ventas.txt",
        "3": "historial_clientes.txt",
        "4": "log_errores.txt"
    }
    
    for num, nombre_arch in archivos_disponibles.items():
        print(num + ". " + nombre_arch)
        
    eleccion = input("\nEscribe el numero del archivo que deseas auditar/leer: ")
    
    if eleccion in archivos_disponibles:
        archivo_elegido = archivos_disponibles[eleccion]
        try:
            with open(archivo_elegido, "r", encoding="utf-8") as arch:
                print("\n--- LEYENDO: " + archivo_elegido + " ---")
                contenido = arch.read()
                if contenido == "":
                    print("(El archivo esta completamente vacio)")
                else:
                    print(contenido)
        except FileNotFoundError:
            print("\nAviso: El archivo '" + archivo_elegido + "' no se encuentra en la carpeta.")
    else:
        print("\nOpcion incorrecta.")
        
    input("\nPresiona ENTER para regresar al menu principal...")

# ==================== FUNCION PRINCIPAL DEL SISTEMA ====================
def main():
    # Creamos y configuramos los archivos .txt automáticamente al arrancar
    inicializar_archivos()
    
    limpiar_pantalla()
    
    # Pedimos el nombre para la bienvenida
    print("========================================")
    nickname = input("Ingrese su nombre o nickname de cajero: ")
    print("========================================")
    
    bienvenida = "¡Hola, " + nickname + "! Bienvenido al sistema de cobro."
    print(bienvenida)
    
    pantalla_carga()
    limpiar_pantalla()
    
    # Guardamos la fecha del turno en una tupla
    print("--- Configuracion del Turno ---")
    dia = input("Dia de hoy (ej. 25): ")
    mes = input("Mes actual (ej. 09): ")
    anio = input("Anio en curso (ej. 2026): ")
    
    Fecha = dia, mes, anio  
    fecha_str = Fecha[0] + "/" + Fecha[1] + "/" + Fecha[2]
    print("Fecha de operacion guardada:", Fecha, "\n")
    time.sleep(1.5) 

    # Menú principal como una lista de listas
    matriz_menu = [
        ["1. Cobrar pedido", "2. Impresion de tickets"],
        ["3. Perfil Usuario", "4. Salir del sistema"]
    ]

    continuar = True
    
    while continuar:
        limpiar_pantalla()
        print("="*45)
        print("               MENU PRINCIPAL              ")
        print("="*45)
        
        for fila in matriz_menu:
            print(fila[0] + "   |   " + fila[1])
        print("="*45)
        
        # Control de inactividad
        inicio_tiempo = time.time()
        opcion = input("\nSeleccione una opcion del menu (1-4): ")
        fin_tiempo = time.time()
        
        tiempo_espera = fin_tiempo - inicio_tiempo
        
        if tiempo_espera > 600:
            for i in range(1):
                print("\n*** AVISO: Detectamos mas de 10 minutos de inactividad ***")
                respuesta = input("El menu se suspendio. ¿Desea continuar? (si/no): ")
                if respuesta.lower() != "si":
                    print("Saliendo a la pantalla de inicio...")
                    continuar = False
            
            if not continuar:
                break 

        # Redirigir según la opción elegida
        if opcion == "1":
            realizar_cobro(nickname, fecha_str)
        elif opcion == "2":
            imprimir_tickets()
        elif opcion == "3":
            area_usuario()
        elif opcion == "4":
            print("\nCerrando el sistema de cajas... ¡Excelente turno, " + nickname + "!")
            time.sleep(2)
            limpiar_pantalla()
            continuar = False
        else:
            print("\nOpcion no valida. Teclea un numero del 1 al 4.")
            time.sleep(1)

if __name__ == "__main__":
    main()

¡Hola, d! Bienvenido al sistema de cobro.

Iniciando conexion con el servidor de Carl's Jr...
Cargando bases de datos de hamburguesas...
¡Sistema listo y cargado!

--- Configuracion del Turno ---
Fecha de operacion guardada: ('12', '12', '12') 

               MENU PRINCIPAL              
1. Cobrar pedido   |   2. Impresion de tickets
3. Perfil Usuario   |   4. Salir del sistema
--- NUEVA ORDEN ---

--- MENU DE HAMBURGUESAS ---
1. Famous Star - $110.00
2. Western Bacon - $135.00
3. Guacamole Burger - $150.00
4. Combo Super Star - $180.00
-> ¡Agregado al pedido!
-> ¡Agregado al pedido!

--- PROCESO DE PAGO ---
El total a cobrar es de: $504.6
Por favor, inserte o deslice la tarjeta en la terminal...
Procesando...
¡Pago aprobado! Retirando tarjeta...

Fecha: 12/12/12 | Cajero: d
Cliente a llamar: JUAN
----------------------------------------
- Western Bacon x1 ($135.0)
- Guacamole Burger x2 ($300.0)
----------------------------------------
Subtotal: $435.0
IVA (16%): $69.6
Total Final: $5